# Notebook 4: Building Your Own Environmental Model

## Learning Goals
- Understand the structure of environmental models
- Learn key concepts: state variables, parameters, driving data
- Build a simple plant growth model from scratch
- Evaluate model performance
- **Prepare to build the SWBM yourself**

---

## What is an Environmental Model?

An environmental model simulates how nature changes over time. Every model has three key components:

**1. State Variables** - Things that change (what we're tracking)
- Examples: soil moisture, plant biomass, water level, temperature

**2. Parameters** - Fixed properties of the system
- Examples: maximum growth rate, water holding capacity, evaporation rate

**3. Driving Data** - External inputs that change over time
- Examples: precipitation, temperature, solar radiation

The model equation shows how state variables change:
$$X_{t+1} = X_t + f(X_t, \text{Parameters}, \text{Driving Data})$$

---

## 1. Building a Simple Plant Growth Model

Let's build a model where plant biomass grows based on temperature and water availability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Define the system components

# Parameters (fixed properties of our plant)
params = {
    'max_growth_rate': 0.02,      # Maximum growth per day (fraction of current biomass)
    'optimal_temp': 20,            # Optimal temperature in °C
    'temp_range': 10,              # Temperature range around optimal (°C)
    'water_demand': 50,            # Water needed for max growth (mm)
    'max_biomass': 2000            # Maximum possible biomass (g/m²)
}

# State variable
initial_biomass = 50  # g/m² (seedling)

# Driving data - temperature and rainfall for 100 days
days = 100
driving_data = {
    'day': np.arange(days),
    'temperature': 15 + 10 * np.sin(np.arange(days) / days * 2 * np.pi),  # Seasonal variation
    'rainfall': np.random.gamma(shape=2, scale=5, size=days)  # Random rainfall events
}

print("Model Components:")
print(f"\nParameters:")
for param, value in params.items():
    print(f"  {param}: {value}")
print(f"\nInitial Biomass: {initial_biomass} g/m²")
print(f"\nDriving Data:")
print(f"  Temperature range: {driving_data['temperature'].min():.1f}°C to {driving_data['temperature'].max():.1f}°C")
print(f"  Total rainfall: {driving_data['rainfall'].sum():.1f} mm")

## 2. Write Functions for Each Process

Break the model into individual functions, each calculating one process:

In [ ]:
def calculate_growth_limitation_temperature(temp, optimal_temp, temp_range):
    """
    Calculate how temperature limits plant growth.
    Growth is maximum at optimal temp and decreases away from it.
    
    Returns a fraction 0 to 1.
    """
    # Distance from optimal temperature
    temp_stress = abs(temp - optimal_temp) / temp_range
    
    # If too far from optimal, no growth
    if temp_stress > 1:
        return 0
    
    # Otherwise, stress reduces growth (Gaussian-like response)
    growth_factor = 1 - (temp_stress ** 2)
    return growth_factor

def calculate_growth_limitation_water(rainfall, water_demand):
    """
    Calculate how water limits plant growth.
    More water = more growth, but there's a maximum.
    
    Returns a fraction 0 to 1.
    """
    water_factor = min(rainfall / water_demand, 1.0)
    return water_factor

def calculate_growth(current_biomass, max_growth_rate, max_biomass, 
                    temp_factor, water_factor):
    """
    Calculate daily biomass growth.
    
    Growth depends on:
    - Size (limited by max_biomass)
    - Growth rate
    - Both temperature and water availability
    """
    # Space available for growth
    space_factor = 1 - (current_biomass / max_biomass)
    
    # All three must be favorable for growth
    combined_factor = temp_factor * water_factor * space_factor
    
    # Growth amount (fraction of current biomass)
    growth = current_biomass * max_growth_rate * combined_factor
    
    return growth

# Test the functions
print("Testing functions:")
print(f"\nTemperature factor at 20°C (optimal): {calculate_growth_limitation_temperature(20, 20, 10):.3f}")
print(f"Temperature factor at 10°C (cold): {calculate_growth_limitation_temperature(10, 20, 10):.3f}")
print(f"Temperature factor at 30°C (hot): {calculate_growth_limitation_temperature(30, 20, 10):.3f}")

print(f"\nWater factor at 50mm (optimal): {calculate_growth_limitation_water(50, 50):.3f}")
print(f"Water factor at 25mm (dry): {calculate_growth_limitation_water(25, 50):.3f}")
print(f"Water factor at 100mm (wet): {calculate_growth_limitation_water(100, 50):.3f}")

## 3. Run the Model - Step-by-Step Simulation

Now simulate the plant growing over 100 days:

In [ ]:
# Run the simulation
biomass = np.zeros(days + 1)
biomass[0] = initial_biomass

# Storage for intermediate values
temp_factors = np.zeros(days)
water_factors = np.zeros(days)
growth_amounts = np.zeros(days)

# Simulation loop
print(f"Day | Temp(°C) | Rain(mm) | TempFact | WaterFact | Growth(g) | Biomass(g)")
print("-" * 75)

for day in range(days):
    # Get driving data
    temp = driving_data['temperature'][day]
    rain = driving_data['rainfall'][day]
    
    # Calculate limiting factors
    temp_factor = calculate_growth_limitation_temperature(temp, params['optimal_temp'], params['temp_range'])
    water_factor = calculate_growth_limitation_water(rain, params['water_demand'])
    
    # Calculate growth
    growth = calculate_growth(biomass[day], params['max_growth_rate'], params['max_biomass'],
                             temp_factor, water_factor)
    
    # Update biomass
    biomass[day + 1] = biomass[day] + growth
    
    # Store for analysis
    temp_factors[day] = temp_factor
    water_factors[day] = water_factor
    growth_amounts[day] = growth
    
    # Print selected days
    if day % 10 == 0 or day < 3:
        print(f"{day+1:3d} | {temp:7.1f} | {rain:7.1f} | {temp_factor:8.3f} | {water_factor:9.3f} | {growth:9.1f} | {biomass[day+1]:9.1f}")

print(f"\nSimulation Summary:")
print(f"Starting biomass: {biomass[0]:.1f} g/m²")
print(f"Ending biomass: {biomass[-1]:.1f} g/m²")
print(f"Total growth: {biomass[-1] - biomass[0]:.1f} g/m²")
print(f"Mean daily growth: {growth_amounts.mean():.2f} g/m²")

## 4. Visualize Model Output

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9))

days_array = np.arange(days + 1)

# Panel 1: Driving data
ax1 = axes[0]
ax1_twin = ax1.twinx()
ax1.bar(days_array[:-1], driving_data['rainfall'], color='blue', alpha=0.5, label='Rainfall')
ax1_twin.plot(days_array[:-1], driving_data['temperature'], color='red', linewidth=2, label='Temperature')
ax1.set_ylabel('Rainfall (mm)', color='blue')
ax1_twin.set_ylabel('Temperature (°C)', color='red')
ax1.set_title('Driving Data: Temperature and Rainfall')
ax1.grid(alpha=0.3)

# Panel 2: Plant biomass
ax2 = axes[1]
ax2.plot(days_array, biomass, linewidth=2, color='green', label='Biomass')
ax2.axhline(params['max_biomass'], color='red', linestyle='--', alpha=0.5, label='Maximum capacity')
ax2.set_ylabel('Biomass (g/m²)')
ax2.set_title('Plant Growth Over Time')
ax2.legend()
ax2.grid(alpha=0.3)

# Panel 3: Limiting factors
ax3 = axes[2]
ax3.plot(days_array[:-1], temp_factors, label='Temperature factor', linewidth=1.5)
ax3.plot(days_array[:-1], water_factors, label='Water factor', linewidth=1.5)
ax3.set_xlabel('Days')
ax3.set_ylabel('Growth Limitation Factor (0-1)')
ax3.set_title('What Limits Plant Growth?')
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Model outputs visualized!")

## 5. Parameter Sensitivity - What Matters Most?

Let's see how sensitive the model is to different parameters:

In [ ]:
def run_model_sensitivity(params_base, driving_data, param_to_vary, variation_values):
    """
    Run the model multiple times with different parameter values.
    
    Returns final biomass for each variation.
    """
    results = []
    
    for variation in variation_values:
        # Copy parameters and modify one
        params_test = params_base.copy()
        params_test[param_to_vary] = variation
        
        # Run model
        biomass = params_base['max_biomass'] * 0.025  # Small initial value
        
        for day in range(len(driving_data['temperature'])):
            temp = driving_data['temperature'][day]
            rain = driving_data['rainfall'][day]
            
            temp_factor = calculate_growth_limitation_temperature(temp, params_test['optimal_temp'], params_test['temp_range'])
            water_factor = calculate_growth_limitation_water(rain, params_test['water_demand'])
            
            growth = calculate_growth(biomass, params_test['max_growth_rate'], params_test['max_biomass'],
                                     temp_factor, water_factor)
            biomass = biomass + growth
        
        results.append(biomass)
    
    return results

# Test sensitivity to max_growth_rate
growth_rates = np.linspace(0.01, 0.05, 5)
results_growth = run_model_sensitivity(params, driving_data, 'max_growth_rate', growth_rates)

# Test sensitivity to water_demand
water_demands = np.linspace(20, 100, 5)
results_water = run_model_sensitivity(params, driving_data, 'water_demand', water_demands)

# Plot sensitivity
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(growth_rates, results_growth, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Maximum Growth Rate')
axes[0].set_ylabel('Final Biomass (g/m²)')
axes[0].set_title('Sensitivity: How Growth Rate Affects Final Biomass')
axes[0].grid(alpha=0.3)

axes[1].plot(water_demands, results_water, 'o-', color='blue', linewidth=2, markersize=8)
axes[1].set_xlabel('Water Demand (mm)')
axes[1].set_ylabel('Final Biomass (g/m²)')
axes[1].set_title('Sensitivity: How Water Requirement Affects Final Biomass')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSensitivity Analysis:")
print(f"Growth rate impact: Final biomass ranges from {min(results_growth):.0f} to {max(results_growth):.0f} g/m²")
print(f"Water demand impact: Final biomass ranges from {min(results_water):.0f} to {max(results_water):.0f} g/m²")

## 6. Understanding Model Limitations

Every model has limitations. Let's discuss what our plant model doesn't include:

In [1]:
limitations = {
    'What we model': [
        'Total plant biomass (one pool)',
        'Temperature effect on growth',
        'Water availability effect on growth',
        'Maximum growth capacity'
    ],
    'What we ignore': [
        'Nutrient availability (N, P, K)',
        'Pests and diseases',
        'Allocation to roots vs leaves vs stems',
        'CO₂ concentration effects',
        'Day length (photoperiod) effects',
        'Soil texture and structure',
        'Plant age/developmental stage'
    ],
    'Which simplifications matter?': [
        'For predictions over 100 days: probably OK',
        'For predictions over 10 years: missing nutrient cycling!',
        'For different regions: likely need location-specific parameters',
        'For different species: completely different model needed'
    ]
}

print("Model Assumptions and Limitations\n" + "="*50)
for category, items in limitations.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  • {item}")

print("\n" + "="*50)
print("Key lesson: All models are simplifications!")
print("The art of modeling is choosing which details matter.")

Model Assumptions and Limitations

What we model:
  • Total plant biomass (one pool)
  • Temperature effect on growth
  • Water availability effect on growth
  • Maximum growth capacity

What we ignore:
  • Nutrient availability (N, P, K)
  • Pests and diseases
  • Allocation to roots vs leaves vs stems
  • CO₂ concentration effects
  • Day length (photoperiod) effects
  • Soil texture and structure
  • Plant age/developmental stage

Which simplifications matter?:
  • For predictions over 100 days: probably OK
  • For predictions over 10 years: missing nutrient cycling!
  • For different regions: likely need location-specific parameters
  • For different species: completely different model needed

Key lesson: All models are simplifications!
The art of modeling is choosing which details matter.


## Exercises

### Exercise 1: Modify the Model
Add a new process to the plant growth model:
- Implement plant mortality/senescence: plants lose biomass when too cold (temp < 5°C)
- Create a function `calculate_mortality(temperature, biomass)`
- Integrate it into the simulation loop

In [ ]:
# YOUR CODE HERE
# def calculate_mortality(temperature, biomass, death_threshold):
#     """
#     Calculate how much biomass is lost in cold conditions.
#     """
#     if temperature < death_threshold:
#         mortality = ...
#     else:
#         mortality = 0
#     return mortality

# # Then incorporate into the main simulation loop...

### Exercise 2: Build a Simple Lake Temperature Model
Create a new model (similar structure to what we did):
- **State variable:** Lake surface temperature
- **Driving data:** Air temperature
- **Process:** Water warms/cools toward air temperature 
- **Parameter:** Heat exchange rate (how fast water reaches air temperature)

Then run it and plot the results.

In [ ]:
# YOUR CODE HERE
# Define parameters, initial state, and driving data
# Write a function to calculate temperature change
# Run a simulation loop
# Plot water temperature vs air temperature

## Summary

You now understand:
- ✓ The structure of environmental models
- ✓ State variables, parameters, and driving data
- ✓ How to build a model from scratch
- ✓ How to run simulations and visualize results
- ✓ How to analyze model sensitivity
- ✓ Model limitations and assumptions

**You're now ready to understand ANY environmental model - including the SWBM!**

## Your Next Challenge

Build the **Soil Water Balance Model** yourself:
1. Look at the data structure the SWBM expects
2. Define the state variable (soil moisture)
3. Write functions for each process (ET, runoff, infiltration)
4. Build the main simulation loop
5. Evaluate your model against observations

You have all the skills you need. This is now in your hands! 🚀